In [23]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages.utils import trim_messages,count_tokens_approximately
import os

In [24]:
load_dotenv()  # Load environment variables from .env file

True

In [25]:
google_api_key = os.getenv("GOOGLE_API_KEY")
if not google_api_key:
    raise ValueError("GOOGLE_API_KEY is missing")

postgres_uri = os.getenv("POSTGRES_URI")
if not postgres_uri:
    raise ValueError("POSTGRES_URI is missing")

In [26]:
primary = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=google_api_key,
    max_retries=3,
)
backup = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=google_api_key,
    max_retries=3,
)

model = primary.with_fallbacks([backup])

In [27]:
class MemoryState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [28]:
MAX_TOKENS = 150

In [29]:
def chat_model(state: MemoryState):
    # Trim messages to fit within the token limit
    messages = trim_messages(
        state["messages"],
        max_tokens = MAX_TOKENS,  # Set your desired token limit
        token_counter = count_tokens_approximately, # Use the approximate token counting function
        strategy = "last"  # Keep the last messages within the limit
    )

    print(f"Trimmed messages to fit within {MAX_TOKENS} tokens:")
    for msg in messages:
        print(f"{msg.type}: {msg.content}")

    response = model.invoke(messages)

    return {"messages": [response]}


In [30]:
builder = StateGraph(MemoryState)
builder.add_node("call_model", chat_model)
builder.add_edge(START, "call_model")

In [31]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [32]:
config = {"configurable": {"thread_id": "chat-1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Nitish."}]},
    config,
)

result["messages"][-1].content

Trimmed messages to fit within 150 tokens:
human: Hi, my name is Nitish.


[{'type': 'text',
  'text': 'Hello Nitish! Nice to meet you. How can I help you today?',
  'extras': {'signature': 'ErIHCq8HAWkUfRNSi/y0MCJlu4BWNm6ZRbNYPBqgeDXyi6TibMAdI0+gLaYeO9RfCl6/uU5cMYRQKSN+prXJeNvI/fmoF4Ko/AcaaI3eyt1yQ9uzXDaX7WctfJJcFFHlIu8JOWxxFNfYclqcNcnlUKH0l1hTZUY7+wJEbprDPuiWUi8NgIyT6sq69pmzgoGTEArYhSqMt8ovjv4vl49Cj9but8sBeXGxli0X13yrCNXzYpZA+kdJJvzawDTtvIcoE+eKbtYMxab2EzWGrP6UrH+z5mBfa+eYKcXOxhykKP4qJKgxGeZyVT/yzBwdYUC2vMkqJErUEkzb3Ckn4FCBBTtvAyKQa2pf1G/hPKzrqS6QDSpvfazBZAlV3ZKE/DmUUVmBeIRIBsFgJ0kPGAznq5/QT65BbapxhlvjRspBUxFUH1jZ2fVDsy4do7l8Tn3rzXZD9nLBFTR0Gyl2CMXDUc1Dq56+qk+FATuz3NwRAWee9eCXDEoBgI9QxH5Op6R4NukFgIuF2YdnKdZ8oiQ47DV6BndFC1tj3OZxGFw7DiutAruPXkMGn4+XpEHqirwDXOTKGVkPHwt2WvuCfxbMw1pAEeTQoW/ClPmUkjrOgF29Fe0d2TTK0Y/JH/C3e7NNtcN5hNBDkjtaNyTlMZuR7ST65HU4i89TcUBr5DGU3hHcZPXBLZOlWDeL6ih6lj77Ds5HeKWFA/whbozrVsyL++4XKVGOS7vcMekX8kne13zpDstcd8L0uv08ucbwB4+DJkZJ4LPklITrfdppjnmsF8xKoKrcwGcIx/0BJnOLUJh0LBrO4sHqZSYJFMiy9FH2X7HB39m/mvKVukXGArwZ50EKb6nRd8/ol8sdGUmGvsBwB8rceuQG

In [33]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "I am learning LangGraph."}]},
    config,
)

result["messages"][-1].content

Trimmed messages to fit within 150 tokens:
human: Hi, my name is Nitish.
ai: [{'type': 'text', 'text': 'Hello Nitish! Nice to meet you. How can I help you today?', 'extras': {'signature': 'ErIHCq8HAWkUfRNSi/y0MCJlu4BWNm6ZRbNYPBqgeDXyi6TibMAdI0+gLaYeO9RfCl6/uU5cMYRQKSN+prXJeNvI/fmoF4Ko/AcaaI3eyt1yQ9uzXDaX7WctfJJcFFHlIu8JOWxxFNfYclqcNcnlUKH0l1hTZUY7+wJEbprDPuiWUi8NgIyT6sq69pmzgoGTEArYhSqMt8ovjv4vl49Cj9but8sBeXGxli0X13yrCNXzYpZA+kdJJvzawDTtvIcoE+eKbtYMxab2EzWGrP6UrH+z5mBfa+eYKcXOxhykKP4qJKgxGeZyVT/yzBwdYUC2vMkqJErUEkzb3Ckn4FCBBTtvAyKQa2pf1G/hPKzrqS6QDSpvfazBZAlV3ZKE/DmUUVmBeIRIBsFgJ0kPGAznq5/QT65BbapxhlvjRspBUxFUH1jZ2fVDsy4do7l8Tn3rzXZD9nLBFTR0Gyl2CMXDUc1Dq56+qk+FATuz3NwRAWee9eCXDEoBgI9QxH5Op6R4NukFgIuF2YdnKdZ8oiQ47DV6BndFC1tj3OZxGFw7DiutAruPXkMGn4+XpEHqirwDXOTKGVkPHwt2WvuCfxbMw1pAEeTQoW/ClPmUkjrOgF29Fe0d2TTK0Y/JH/C3e7NNtcN5hNBDkjtaNyTlMZuR7ST65HU4i89TcUBr5DGU3hHcZPXBLZOlWDeL6ih6lj77Ds5HeKWFA/whbozrVsyL++4XKVGOS7vcMekX8kne13zpDstcd8L0uv08ucbwB4+DJkZJ4LPklITrfdppjnmsF8xKoKrcwGcIx/0BJnOLUJh

[{'type': 'text',
  'text': "That's great, Nitish! LangGraph is a fantastic framework for building complex, stateful, and multi-agent applications with LLMs. It excels at managing workflows with loops, persistence, and human-in-the-loop interactions.\n\nWhether you're just starting out or working on a specific project, I can help you with:\n\n* **Core Concepts:** Understanding `StateGraph`, `Nodes`, `Edges`, `State`, and `Conditional Edges`.\n* **Agent Patterns:** Building multi-agent systems, supervisor models, or self-correcting RAG pipelines.\n* **Advanced Features:** Setting up checkpointers (persistence), human-in-the-loop approvals, and time-travel debugging.\n* **Code Examples:** Writing or debugging Python code for your graphs.\n\nWhere are you currently at in your learning, or is there a specific concept or project you'd like to work on right now?",
  'extras': {'signature': 'Es4RCssRAWkUfROargCfrw6kHbm23wPJA9367i1w0mLDnpNVUN+SXRODiJ2wZJiGvutdNlk9P6lYjkSCoOzBsjWGjkroeubO9+4/QH

In [34]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "Can you explain short term memory?"}]},
    config,
)

result["messages"][-1].content

Trimmed messages to fit within 150 tokens:
human: Can you explain short term memory?


[{'type': 'text',
  'text': 'Think of **short-term memory (STM)** as your brain’s temporary "scratchpad" or a mental sticky note. It is the system your brain uses to hold a small amount of information in an active, readily available state for a brief period of time. \n\nWithout short-term memory, you wouldn\'t be able to hold a conversation, do mental math, or remember the beginning of a sentence by the time you reach the end.\n\nHere is a breakdown of how short-term memory works, its limitations, and how it differs from other types of memory.\n\n---\n\n### 1. Key Characteristics of Short-Term Memory\n\nShort-term memory has two major limitations: **duration** and **capacity**.\n\n*   **Brief Duration:** Information in short-term memory typically lasts only **15 to 30 seconds**. If you don\'t actively repeat the information or transfer it to long-term memory, it fades away completely.\n*   **Limited Capacity:** Short-term memory can only hold a few items at once. \n    *   In 1956, psy

In [35]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config,
)

result["messages"][-1].content

Trimmed messages to fit within 150 tokens:
human: What is my name?


[{'type': 'text',
  'text': "I don't know your name yet, as you haven't told me! What should I call you?",
  'extras': {'signature': 'EpoGCpcGAWkUfRP86ADB2lPhNmWMQFh7sIVWC3a6g1YIaPZOfBsiBZSCK9xOOdCoaf6/dkyH8/AeyUOdwY95sFgj8P+EUavn0lmiNHPOPpX5wQZOLYZqCl5EuEAsRoGfRfQ2kYtQmseJBegAO2IswB0Jr5F8p6qM6aeDgWWFxcVAiuN/Bwl3Ad28LWYbKPa8FF+HWNA5/VQeSrdY1xYChHfELfplCiMn5PkCSXDBfRCZxHOQoqxwuaqep6JkisXDSDxo78/BEmKyARZbolL4dpxWnk6YEdTXQT2aBB6xaZAGk3YnAb5/yNARBImuJ+IawhcicZ+LU9B9sbPab0yWI+KtxRQiYd+ersXyBR/0+ZZXutUDCj0SclSapgvEVtEOkJ2C/wE1oztjAgk8dSPih0ZYrWkoeqDvCTR+hDKwdwnsGWGtc6IU6MlhwBow7wNtgOX9x2exjI/kd1TSt3R/Tt0NmvDZBVOIs1THrR2vIodF5e+lomB2hx4Gz5QCUHm6zyqHtWDapkfgy58kuLW3XOTEeCjbg2E8J21QgCLDjkm1tBqKcG71KcA6upQVc/1xouY5H9fgihMTX87dZXZlyR4MDelpaj1t1U72n4oyQSvI1Q+Prwt8hcRzoiHeqjBd2z6SmLudt2xkTzgdJ7gwak2+5i8GmMcSZHjp8cVOBi7RSc3ZZ7RLmALe0FOS2j8tj1jF2LiKooE0pk0efHWq/Ohx4sF+GL8lQyAv9bP6k4FEc1+uMeCDjRrWZbI8EeVMnIR99/RfELv6RS8iS93LJOtnaeu02GFjLVhyPP5O6Ktxz+jYF90rIHll1yIJI4ZMHmVpVtiaTJq5VCTp3JQ5C4qbNfR9CecG3+

In [36]:
for item in graph.get_state({"configurable": {"thread_id": "chat-1"}}).values['messages']:
    print(item.content)
    print('-'*120)

Hi, my name is Nitish.
------------------------------------------------------------------------------------------------------------------------
[{'type': 'text', 'text': 'Hello Nitish! Nice to meet you. How can I help you today?', 'extras': {'signature': 'ErIHCq8HAWkUfRNSi/y0MCJlu4BWNm6ZRbNYPBqgeDXyi6TibMAdI0+gLaYeO9RfCl6/uU5cMYRQKSN+prXJeNvI/fmoF4Ko/AcaaI3eyt1yQ9uzXDaX7WctfJJcFFHlIu8JOWxxFNfYclqcNcnlUKH0l1hTZUY7+wJEbprDPuiWUi8NgIyT6sq69pmzgoGTEArYhSqMt8ovjv4vl49Cj9but8sBeXGxli0X13yrCNXzYpZA+kdJJvzawDTtvIcoE+eKbtYMxab2EzWGrP6UrH+z5mBfa+eYKcXOxhykKP4qJKgxGeZyVT/yzBwdYUC2vMkqJErUEkzb3Ckn4FCBBTtvAyKQa2pf1G/hPKzrqS6QDSpvfazBZAlV3ZKE/DmUUVmBeIRIBsFgJ0kPGAznq5/QT65BbapxhlvjRspBUxFUH1jZ2fVDsy4do7l8Tn3rzXZD9nLBFTR0Gyl2CMXDUc1Dq56+qk+FATuz3NwRAWee9eCXDEoBgI9QxH5Op6R4NukFgIuF2YdnKdZ8oiQ47DV6BndFC1tj3OZxGFw7DiutAruPXkMGn4+XpEHqirwDXOTKGVkPHwt2WvuCfxbMw1pAEeTQoW/ClPmUkjrOgF29Fe0d2TTK0Y/JH/C3e7NNtcN5hNBDkjtaNyTlMZuR7ST65HU4i89TcUBr5DGU3hHcZPXBLZOlWDeL6ih6lj77Ds5HeKWFA/whbozrVsyL++4XKVGOS7vcMekX8kne